# FaceAttend GPU environment verification
    ## Objective
    Prove that this notebook kernel uses the local NVIDIA GPU. Select the project `.venv` in VS Code before running.
    The first executable cell hard fails if CUDA is inaccessible. CPU preprocessing and logic tests are allowed;
    training, smoke-training and inference benchmarking must use CUDA.
    A command-line notebook run proves that kernel's computation; it does not prove VS Code selected that kernel.

In [ ]:
import sys
import torch
import torchvision
print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU-ONLY TRAINING: CUDA unavailable in this kernel. Fix NVIDIA driver, CUDA wheel, interpreter or notebook kernel. CPU training is forbidden.")
device = torch.device("cuda:0")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU properties:", torch.cuda.get_device_properties(0))
x = torch.randn((128, 128), device=device)
y = x @ x
assert y.is_cuda and y.device.type == "cuda"
torch.cuda.synchronize()
print("GPU tensor test PASSED:", y.device)
del x, y

## Record the verified environment
Save machine/software metadata locally for subsequent experiment provenance.

In [ ]:
from pathlib import Path
import os, sys
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), None)
if ROOT is None:
    raise RuntimeError("Open the FaceAttend project folder.")
sys.path.insert(0, str(ROOT / "src"))
os.environ["TORCH_HOME"] = str(ROOT / ".cache/torch")
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from faceattend.utils import environment, load_config, seed_everything, write_json

In [ ]:
report = environment()
write_json(ROOT / "outputs/environment_check.json", report)
display(pd.Series(report, name="Verified notebook environment"))

## Interpretation and next step
    A passed tensor test demonstrates CUDA computation in the printed Python interpreter. Confirm it ends in
    `.venv/Scripts/python.exe`. Run this cell in VS Code itself before starting either training notebook.
    Driver compatibility and installed wheel versions must be retained with each run; `nvidia-smi` alone is insufficient.